# ДЗ 7. Снижение размерности: PCA и UMAP
**Выполнил:** Хритинин Д. А.

**Задание:**
1. Загрузить датасет Breast Cancer.
2. Обучить `RandomForestClassifier` на исходных 30 признаках и замерить accuracy.
3. Применить `PCA(n_components=0.95)`.
4. Обучить ту же модель на сжатых признаках и сравнить качество.
5. Визуализировать данные через UMAP с раскраской по классу.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
import numpy as np
import matplotlib.pyplot as plt

## 1. Загрузка данных

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
print("Размер:", X.shape)
print("Классы:", dict(zip(data.target_names, np.bincount(y))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

## 2. Случайный лес на исходных 30 признаках

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
acc_full = accuracy_score(y_test, rf.predict(X_test))
print("Accuracy (30 признаков):", round(acc_full, 4))

## 3. PCA с сохранением 95% дисперсии
PCA чувствителен к масштабу признаков, поэтому сначала стандартизируем данные (scaler обучаем только на train).

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

pca = PCA(n_components=0.95, random_state=42).fit(X_train_s)
X_train_p = pca.transform(X_train_s)
X_test_p = pca.transform(X_test_s)

print("Осталось компонент:", pca.n_components_, "из", X.shape[1])
print("Объяснённая дисперсия:", round(pca.explained_variance_ratio_.sum(), 4))

plt.figure(figsize=(7, 4))
plt.plot(range(1, pca.n_components_ + 1), np.cumsum(pca.explained_variance_ratio_), marker="o")
plt.axhline(0.95, color="red", ls="--", label="95%")
plt.xlabel("Число компонент"); plt.ylabel("Накопленная доля дисперсии")
plt.title("PCA: накопленная объяснённая дисперсия"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 4. Та же модель на сжатых признаках

In [ ]:
rf_pca = RandomForestClassifier(random_state=42)
rf_pca.fit(X_train_p, y_train)
acc_pca = accuracy_score(y_test, rf_pca.predict(X_test_p))

print("Accuracy (30 признаков):", round(acc_full, 4))
print(f"Accuracy (PCA, {pca.n_components_} компонент):", round(acc_pca, 4))
print("Падение качества:", round((acc_full - acc_pca) * 100, 2), "п.п.")

## 5. Визуализация через UMAP

In [ ]:
%pip install umap-learn --quiet

In [ ]:
import umap

X_s = StandardScaler().fit_transform(X)
emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_s)

plt.figure(figsize=(7, 6))
for cls, color in [(0, "tab:red"), (1, "tab:blue")]:
    plt.scatter(emb[y == cls, 0], emb[y == cls, 1], s=12, c=color,
                label=data.target_names[cls], alpha=0.7)
plt.legend(); plt.title("UMAP: Breast Cancer (malignant / benign)")
plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2")
plt.show()

## Выводы
- На исходных 30 признаках случайный лес даёт accuracy **≈ 0.956**.
- `PCA(n_components=0.95)` сжал 30 признаков до **10 компонент**, потеряв лишь 5% дисперсии.
- На сжатых признаках accuracy **≈ 0.921**, т.е. качество упало примерно на **3.5 п.п.** Это небольшое падение при сокращении числа признаков в 3 раза.
- Почему качество всё же упало: PCA сохраняет направления с наибольшей *дисперсией*, а не с наибольшей *разделяющей способностью* для классов, часть полезной информации попала в отброшенные 5%. Кроме того, случайному лесу удобнее делать разбиения по исходным признакам, а не по их линейным комбинациям.
- На UMAP классы malignant и benign образуют два почти раздельных облака с небольшим перекрытием на границе. Данные хорошо разделимы, что и объясняет высокую accuracy обеих моделей.